# Clase 08 · TI-support-RAG con Supabase y Groq
**Adaptación del ejemplo de Robert Garcia Rey · EAM 2026-II**  
Fecha de adaptación: 14 de octubre de 2026.

Seguimos las diez etapas del notebook docente `parcial2/Clase_08_Demo_Colab_RAG_Supabase_Groq.ipynb`.
Este notebook orquesta los módulos del proyecto: no redefine embeddings, permisos, prompts ni validadores.
El corpus es **académico simulado**, no un manual operativo real. Se distingue evidencia histórica,
recuperación real, salidas mock y generación real.

La ejecución predeterminada revisa fuentes, índice y contratos **sin llamar a Supabase/Groq ni cargar el modelo**.
Activa las opciones necesarias en la sección 1 y vuelve a ejecutar desde esa sección.
No se asegura fidelidad universal; la suite remota completa sigue pendiente por cuota.

## 1. Preparación y decisiones de ejecución
Desde la raíz: `source .venv/bin/activate`, `python -m pip install -r requirements.txt`.
Selecciona en Jupyter el Python de `.venv`. El notebook no instala dependencias automáticamente.

A diferencia del ejemplo de Colab, usamos el repositorio y `.env` local. Para Colab se necesita
una copia autorizada del proyecto, sus fuentes e índice, y secretos de sesión; no subas `.env`.
No es un notebook autónomo que funcione solo con el ZIP docente.

Opciones separadas: cargar el embedder; conectar; escribir mediante upsert; consultar Groq;
regresión de diez casos; OCR; comparación; API; tests; PDF. Las opciones de red/escritura
están desactivadas para que Run All no gaste cuota ni modifique Supabase.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib
from datetime import datetime, timezone
from IPython.display import display, Markdown

roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in roots if (p/'src/rag').is_dir() and (p/'manage.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Abre el notebook desde TI-support-RAG o su directorio notebooks.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if Path(sys.prefix).resolve() != (PROJECT_ROOT/'.venv').resolve():
    raise RuntimeError(
        f'Kernel actual: {sys.executable}. Selecciona Python (.venv TI-support-RAG) '
        f'en el selector de kernel y reinicia. Python esperado: {PROJECT_ROOT / ".venv/bin/python"}'
    )

ENABLE_EMBEDDINGS = False
CONNECT_SUPABASE = True
UPLOAD_EXISTING_INDEX = False
RUN_GROQ_QUERY = False
RUN_CLASS08_CASES = False
RUN_REMOTE_REGRESSION = False
COMPARE_CHUNKING = False
RUN_OCR = False
RUN_API_CHECKS = False
RUN_LOCAL_TESTS = False
GENERATE_PDF = False
RUN_LOCAL_MOCK = False
INDEX_PATH = PROJECT_ROOT/'data/indexes/sections'
RUN_DIR = PROJECT_ROOT/'docs/rag_runs/class08'
CONSULTA = 'El portal de documentos devuelve error 403. ¿Eso demuestra que mi cuenta está suspendida?'
OCR_IMAGE = PROJECT_ROOT/'docs/multimodal/multimodal_demo.png'
OCR_ENGINE = 'tesseract'
observed_results = []
persistence_result = None
api_result = None

def save_json(name, value):
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    path = RUN_DIR/name
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    return path

def run_project_module(module, *args):
    # Same kernel/interpreter; no shell, no secrets in arguments/output.
    subprocess.run([sys.executable,'-m',module,*map(str,args)],cwd=PROJECT_ROOT,check=True)

print('Raíz:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('Modo predeterminado: revisión local sin proveedores.')

Raíz: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG
Python: 3.12.3
Modo predeterminado: revisión local sin proveedores.


## 2. Conexión y sesión de práctica
`SupabaseRepository.connect()` carga los secretos solo cuando se usa la integración.
Necesita `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`, `SUPABASE_EMAIL` y `SUPABASE_PASSWORD`.
Auth usa su contraseña propia, distinta de PostgreSQL. No mostrar secretos ni usar `service_role`.
La migración ya aplicada está en `supabase/migrations/20261014_documentary_rag.sql`:
RLS por propietario, sin acceso anónimo ni DELETE; RPC SECURITY INVOKER.
No ejecutamos ni modificamos SQL desde este notebook.

In [3]:
from src.rag.repository import LocalRepository, SupabaseRepository
remote_repository = None
if CONNECT_SUPABASE:
    remote_repository = SupabaseRepository.connect()
    print('Sesión Supabase establecida; identidad y credenciales no se imprimen.')
else:
    print('Conexión remota no ejecutada.')

Conexión remota no ejecutada.


## 3. Abrimos el índice y revisamos el corpus del proyecto
El docente importa un ZIP con nombres de clase 07. Nuestra adaptación reutiliza
`records.json`, `vectors.npy` y `config.json` con el cargador propio y el manifiesto PDF.
Comprueba hashes de fuentes, integridad del índice, dimensión, normas e identidades.
No convierte el ZIP original de manuales en un índice por cambiarle el nombre.
Si falta el índice, reconstruye explícitamente con `python -m scripts.index_manuals`.

In [1]:
from src.rag.config import RAGConfig
from src.rag.ingestion import load_corpus
from src.rag.index import load_index
from src.config import MODEL, PROMPT_PATH, PROMPT_VERSION
from schemas.request_v1 import Categoria
from typing import get_args

manifest_path = RAGConfig().manifest
manifest = json.loads(manifest_path.read_text())
sections = load_corpus(manifest_path)
CORPUS_ID, chunks, vectors, embedding_config = load_index(INDEX_PATH)
print('Categorías:', list(get_args(Categoria)))
print('Documentos:',len(manifest['documents']), '| Secciones:',len(sections))
print('Corpus:',CORPUS_ID,'| Fragmentos:',len(chunks),'| Matriz:',vectors.shape)
print(json.dumps(embedding_config.model_dump(),ensure_ascii=False,indent=2))
display([{'documento':d['document_id'],'categoria':d['category'],'fuente':d['path'],
          'version':d['version'],'simulado':d['simulated']} for d in manifest['documents']])

ModuleNotFoundError: No module named 'src'

## 4. Mantenemos la representación y justificamos la fragmentación
Mismo modelo/revisión que el índice, 384 dimensiones, normalización y coseno.
El límite efectivo es 128 tokens; los fragmentos incluyen encabezados y tienen hasta 80.
`load_index(..., embedder)` recalcula muestras para comprobar compatibilidad.
`sections` es nuestra elección inicial: la comparación no declara un ganador automático.
La evidencia histórica está en `docs/rag_comparison.json`. Repetir la comparación escribe
índices nuevos bajo el directorio de esta sesión, sin reemplazar el índice activo.

In [ ]:
embedder = None
if ENABLE_EMBEDDINGS:
    from src.rag.embeddings import SentenceEmbedder
    embedder = SentenceEmbedder(embedding_config.revision)
    load_index(INDEX_PATH,embedder)
    print('Modelo cargado y muestras verificadas.')
else:
    print('Modelo no cargado; la recuperación requiere ENABLE_EMBEDDINGS=True.')

comparison_path = PROJECT_ROOT/'docs/rag_comparison.json'
comparison = json.loads(comparison_path.read_text()) if comparison_path.is_file() else None
if COMPARE_CHUNKING:
    if embedder is None:
        raise RuntimeError('Activa ENABLE_EMBEDDINGS para comparar.')
    from src.rag.comparison import compare_strategies
    questions = json.loads((PROJECT_ROOT/'cases/rag_questions.json').read_text())
    comparison = compare_strategies(sections,embedder,questions,RUN_DIR/'comparison_indexes')
    save_json('comparison.json',comparison)
if comparison:
    display([{'estrategia':name,'fragmentos':value['count'],
              'tokens_max':value['tokens_max'],'corpus_id':value['corpus_id']}
             for name,value in comparison['strategies'].items()])
    print('Revisar cortes y condiciones antes de justificar una estrategia; evidencia histórica salvo comparación activada.')

## 5. Guardamos fragmentos en Supabase (opcional)
Ya hay una construcción de 70 fragmentos cargada y comprobada. Consultar no exige volver a cargar.
El upsert reutiliza las identidades `(owner_id, corpus_id, chunk_id)`, en lotes de 50;
no borra versiones antiguas. No cambiar arbitrariamente CORPUS_ID.
Se requiere activar tanto la conexión como la escritura y validar antes el índice con el modelo.

In [ ]:
if UPLOAD_EXISTING_INDEX:
    if remote_repository is None or embedder is None:
        raise RuntimeError('Activa CONNECT_SUPABASE y ENABLE_EMBEDDINGS antes de cargar.')
    remote_repository.upsert(CORPUS_ID,chunks,vectors,embedding_config)
    print('Upsert realizado sobre la construcción:',CORPUS_ID)
else:
    print('No se escribieron filas en Supabase.')

## 6. Pregunta nueva y recuperación de candidatos
Modifica `CONSULTA` en la sección 1. `Retriever` combina coseno y BM25, fusión de rangos
y preferencia suave por categoría cuando se proporciona. Top-3 no demuestra suficiencia:
leer el contenido y las condiciones. La similitud no es confianza del clasificador.
Sin conexión usa `LocalRepository` en memoria; con conexión recupera de Supabase.
El embedder rechaza entradas demasiado largas, sin truncarlas silenciosamente.

In [ ]:
from src.rag.retrieval import Retriever
retriever = None
candidates = []
if embedder is not None:
    repository = remote_repository
    if repository is None:
        repository = LocalRepository()
        repository.upsert(CORPUS_ID,chunks,vectors,embedding_config)
    retriever = Retriever(repository,embedder,CORPUS_ID,embedding_config)
    candidates,retrieval_times = retriever.retrieve(CONSULTA)
    display([{'chunk_id':c.chunk_id,'similitud_coseno':c.similitud,
              'fuente':c.metadata['source'],'seccion':c.metadata['section'],
              'paginas':c.metadata['pages'],'texto':c.content} for c in candidates])
    print('Recuperación:', 'Supabase real' if remote_repository else 'índice local real',retrieval_times)
else:
    print('Recuperación no ejecutada: activa ENABLE_EMBEDDINGS.')

## 7. Contrato, prompts, citas y presupuesto
Clasificación `SolicitudTI` y respuesta `RespuestaRAG` son contratos diferentes.
Prompt activo del clasificador: versión system más alta. Prompt documental: RAGConfig.
Se valida estructura, citas existentes y coincidencia con menciones [F1]; eso no basta
para demostrar respaldo semántico. El pipeline añade revisión por modelo y controles locales.

Máximo 1000 tokens de entrada por llamada: mensajes + esquema + reserva 96.
Salida máxima 800 tokens, razonamiento bajo. En generación se reservan 150 tokens
para corrección. El límite de 8000 observado en Groq es TPM, independiente de este presupuesto.

In [ ]:
from schemas.request_v1 import SolicitudTI
from schemas.rag_response_v1 import RespuestaRAG
from src.groq_client import build_payload
from src.token_budget import enforce_input_budget, input_budget
from src.rag.citations import validate_response
from src.rag.context import build_context

print('Clasificador:',PROMPT_VERSION,'| Documental:',RAGConfig().prompt.name,'| Modelo:',MODEL)
print('Contrato documental:',list(RespuestaRAG.model_fields))
classification_budget = enforce_input_budget(build_payload(PROMPT_PATH.read_text(),CONSULTA))
print('Presupuesto estimado clasificación:',classification_budget)
fixture = {'estado':'evidencia_insuficiente','respuesta':'Ejemplo simulado de límite documental.',
           'citas':[],'limitacion':'No se consultó un modelo en esta demostración de contrato.'}
assert validate_response(fixture,{}).estado == 'evidencia_insuficiente'
try:
    validate_response({**fixture,'citas':['F99'],'respuesta':'Afirmación simulada. [F99]'}, {})
except ValueError:
    print('Control negativo: referencia inexistente rechazada.')
else:
    raise AssertionError('Se aceptó una cita fuera del catálogo.')

if candidates:
    response_format={'type':'json_schema','json_schema':{'name':'respuesta_rag','strict':True,
                    'schema':RespuestaRAG.model_json_schema()}}
    prompt=RAGConfig().prompt.read_text()
    context=build_context(CONSULTA,candidates,request_fits=lambda text:
        input_budget(build_payload(prompt,text,response_format=response_format))['total_tokens'] <= 850)
    print('Contexto presupuestado:',context['user_content'])
    print('Omisiones:',context['omitted'])

## 8. Generamos, validamos y mostramos fuentes
`run_rag_case` ejecuta el recorrido construido: clasificación, políticas, recuperación,
pertinencia, contexto, generación, citas, revisión de respaldo y decisión local.
Una consulta real puede consumir varias llamadas y agotar cuota. HTTP 429 es error técnico,
no falta de evidencia. El pipeline conserva trazas y acota esperas/reintentos.

`RUN_GROQ_QUERY` exige conexión real. `RUN_LOCAL_MOCK` usa recuperación real pero
clasificación y respuesta predeterminadas, etiquetadas como mock: no evalúa comprensión.
No activar ambos para atribuir a mocks resultados reales.

In [ ]:
from src.rag.pipeline import run_rag_case
query_result = None
if RUN_GROQ_QUERY or RUN_LOCAL_MOCK:
    if retriever is None:
        raise RuntimeError('Activa ENABLE_EMBEDDINGS.')
    if RUN_GROQ_QUERY and remote_repository is None:
        raise RuntimeError('Activa CONNECT_SUPABASE para la demostración completa remota.')
    mode = 'groq' if RUN_GROQ_QUERY else 'mock'
    case = {'id':'CL08-CONSULTA','tipo':'manual' if mode=='groq' else 'ambiguo','input':CONSULTA}
    query_result = run_rag_case(case,retriever,mode=mode,documentary_fixture=fixture)
    observed_results.append((case,query_result))
    save_json('consulta.json',query_result)
    display({'origen':query_result['source'],'estado_final':query_result['state'],
             'clasificacion':query_result['classification'],
             'documental':query_result['documentary_output'],
             'citas_validas':query_result['rag']['citations_valid'],
             'respaldo':query_result['rag']['semantic_support_review'],
             'etapa_fallo':query_result['failure_stage']})
    if query_result['documentary_output']:
        for label in query_result['documentary_output']['citas']:
            source=query_result['rag']['catalog'][label]
            display({'cita':label,'fuente':source['metadata'],'pasaje':source['content']})
    print('Tiempo observado:',query_result['latency_seconds'],'s; revisión humana de cada afirmación pendiente.')
else:
    print('Generación y mock no ejecutados.')

## 9. Pruebas breves, expectativas y regresión
Expectativas escritas antes de ejecutar. Respondible/paráfrasis: misma regla;
ambiguo: precisión; fuera de cobertura: reconocer límite; dato dinámico: no inventar estado.
Se incluye además seguridad/inyección en la suite de diez casos existente.
La consulta dinámica debe revisarse aparte: el corpus no contiene estados actuales de tickets.

Para la suite real reutilizamos el validador CLI; conserva trazas, detiene los casos pendientes
si recibe cuota bloqueada y termina con error si no todos pasan. No marcar un bloqueo como aprobado.
Las pruebas locales no prueban comprensión del modelo ni RLS entre usuarios reales.

In [ ]:
class08_expectations = [
    {'tipo':'respondible','pregunta':CONSULTA,'esperado':'No inferir suspensión por 403; citar el procedimiento aplicable.'},
    {'tipo':'paráfrasis','pregunta':'Me aparece acceso prohibido 403 en documentos. ¿Mi usuario está suspendido?',
     'esperado':'Conservar significado y recuperar ACS-03.'},
    {'tipo':'ambiguo','pregunta':'No funciona. Ayúdame.','esperado':'Pedir datos que cambian el procedimiento.'},
    {'tipo':'sin cobertura','pregunta':'Dame servidor y puerto exactos de la VPN corporativa.',
     'esperado':'No inventar parámetros ausentes.'},
    {'tipo':'dinámico','pregunta':'¿Mi ticket TI-123 ya fue resuelto hoy?',
     'esperado':'Reconocer que no consulta estados actuales de tickets.'},
]
display(class08_expectations)
regression_cases=json.loads((PROJECT_ROOT/'cases/rag_regression.json').read_text())
print('Suite ampliada:',len(regression_cases),'escenarios con expectativas versionadas.')
if RUN_CLASS08_CASES:
    if retriever is None or remote_repository is None:
        raise RuntimeError('Activa ENABLE_EMBEDDINGS y CONNECT_SUPABASE para las cinco pruebas reales.')
    accepted_states = [ {'respondida','parcial'}, {'respondida','parcial'}, {'aclaracion'},
                        {'evidencia_insuficiente','parcial'}, {'evidencia_insuficiente','parcial'} ]
    class08_checks = []
    for position, (trial, allowed) in enumerate(zip(class08_expectations,accepted_states),1):
        trial_case={'id':f'CL08-PRUEBA-{position}','tipo':'manual','input':trial['pregunta']}
        trial_result=run_rag_case(trial_case,retriever,mode='groq')
        observed_results.append((trial_case,trial_result))
        save_json(f'prueba_{position}.json',trial_result)
        documentary=trial_result['documentary_output'] or {}
        check={'id':trial_case['id'],'tipo':trial['tipo'],'expected':trial['esperado'],
               'state':trial_result['state'],'documentary_state':documentary.get('estado'),
               'no_technical_or_format_error':trial_result['state'] not in {'ERROR_TECNICO','ERROR_FORMATO'},
               'acceptable_documentary_state':documentary.get('estado') in allowed,
               'citations_valid':trial_result['rag']['citations_valid'],
               'support_review':trial_result['rag']['semantic_support_review'],
               'claim_review':'PENDIENTE: contrastar cada afirmación contra el pasaje citado.'}
        class08_checks.append(check)
        save_json('pruebas_clase08.json',class08_checks)
        display(check)
        if 'HTTP 429' in (trial_result['execution_error'] or ''):
            print('Cuota bloqueada: no se ejecutan las pruebas restantes. No se consideran aprobadas.')
            break
else:
    print('Cinco pruebas reales de clase 08 no ejecutadas.')
if RUN_REMOTE_REGRESSION:
    run_project_module('scripts.validate_rag_scenarios','--output',RUN_DIR/'regression')
else:
    print('Regresión remota no ejecutada.')
if RUN_LOCAL_TESTS:
    run_project_module('unittest','discover','-s','test')
    subprocess.run([sys.executable,'manage.py','test','api'],cwd=PROJECT_ROOT,check=True)

## 10. Persistencia y evidencia de la sesión
`verify` abre dos clientes nuevos del mismo usuario, compara inventario y RPC sin upsert
ni recalcular corpus. Usa el vector del primer fragmento como sonda, no una pregunta semántica.
No certifica aislamiento entre dos propietarios; esa prueba sigue pendiente.

Solo se genera PDF si hay resultados de esta sesión. Un PDF mock se rotula como simulado.
El JSON de bitácora guarda opciones efectivamente activadas y pendientes, nunca secretos.
La evidencia histórica se puede leer sin gastar API; no se atribuye a esta sesión.

In [ ]:
if CONNECT_SUPABASE:
    from scripts.verify_supabase import verify
    persistence_result=verify(INDEX_PATH)
    display(persistence_result)
    save_json('persistencia.json',persistence_result)
else:
    print('Persistencia remota no comprobada en esta sesión.')

for name in ('supabase_verification.json','rag_hardening_verification.json','api_validation.json'):
    path=PROJECT_ROOT/'docs'/name
    if path.is_file():
        evidence=json.loads(path.read_text())
        print('Evidencia HISTÓRICA:',name,'| fecha:',evidence.get('timestamp_utc',evidence.get('timestamp','ver archivo')))

if GENERATE_PDF:
    if not observed_results:
        raise RuntimeError('Ejecuta una consulta o mock antes de generar PDF.')
    from src.pdf_report import PDFReport
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    report=PDFReport(RUN_DIR/'clase08.pdf',subtitle='Clase 08 · evidencia de esta sesión (ver origen real/mock)')
    for case,result in observed_results:
        report.register_test_case(case=case,result=result)
    report.generate()
    print('PDF:',report.output_path if hasattr(report,'output_path') else RUN_DIR/'clase08.pdf')

## Anexo A · OCR y fuentes documentales
Reutilizamos OCRFactory/OCRExtractor a través de `prepare_multimodal_input`.
El ejemplo docente incluye lectores PDF/OCR/Excel/Word; nuestro corpus implementado es PDF textual
por secciones y el OCR adjunto es de imágenes de consulta. **No tenemos un ingestor Word/Excel**;
no presentarlos como funcionalidad implementada. `load_corpus` exige páginas textuales válidas;
un manual escaneado requiere extracción OCR revisada y una adaptación de ingestión.
Para probar OCR en el pipeline Groq usa la CLI o el llamado opcional siguiente.

In [ ]:
if RUN_OCR:
    from src.multimodal import prepare_multimodal_input
    from src.ocr import OCRFactory
    print('Motores:',OCRFactory.supported_engines())
    ocr_result=prepare_multimodal_input('La captura muestra el problema.',OCR_IMAGE,ocr_engine=OCR_ENGINE)
    display({'texto_extraido':ocr_result['extracted_text'],'valido':ocr_result['extraction_valid']})
    if RUN_GROQ_QUERY:
        ocr_case={'id':'CL08-OCR','tipo':'multimodal','input':'La captura muestra el problema de conexión.'}
        ocr_rag=run_rag_case(ocr_case,retriever,image_path=OCR_IMAGE,ocr_engine=OCR_ENGINE,mode='groq')
        observed_results.append((ocr_case,ocr_rag))
        save_json('ocr_consulta.json',ocr_rag)
        display({'estado':ocr_rag['state'],'respuesta':ocr_rag['documentary_output']})
else:
    print('OCR no ejecutado. Tesseract requiere el ejecutable y datos de idiomas; EasyOCR puede descargar modelos.')

## Anexo B · API Django y Swagger
API en `api/controllers.py`; cuatro endpoints: health, categories, search y query.
Swagger: http://127.0.0.1:8000/api/v1/docs/ · OpenAPI: `/api/v1/openapi.json`.
Arranca en otro terminal, con `.venv` activo: `python manage.py runserver 127.0.0.1:8000`.
Authorize usa `DJANGO_API_TOKEN`, sin prefijo Bearer; el esquema no contiene el token.
Todos los consumidores usan el mismo usuario Supabase del servidor. La API no tiene OCR HTTP.

Este chequeo opcional hace health, esquema y categorías; no envía query ni consume Groq.

In [ ]:
if RUN_API_CHECKS:
    import requests
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT/'.env')
    base='http://127.0.0.1:8000/api/v1'
    health=requests.get(base+'/health/',timeout=10)
    health.raise_for_status()
    schema=requests.get(base+'/openapi.json',timeout=10)
    schema.raise_for_status()
    token=os.getenv('DJANGO_API_TOKEN','')
    if len(token)<32:
        raise RuntimeError('Configura DJANGO_API_TOKEN localmente; no lo imprimas.')
    categories=requests.get(base+'/categories/',headers={'Authorization':'Bearer '+token},timeout=10)
    categories.raise_for_status()
    api_result={'health':health.json(),'openapi':schema.json()['openapi'],
                'paths':list(schema.json()['paths']),'categories':categories.json()}
    display(api_result)
else:
    print('Comprobación HTTP no ejecutada; abrir Swagger permite explorar contratos y probar operaciones.')

## Cierre · bitácora y revisión humana
Registrar una respuesta útil y un fallo/límite reales, vinculando cada afirmación a su fuente.
Los mocks solo demuestran contratos. Para revisar: ¿llegó la regla al contexto?, ¿cada cita existe?,
¿el pasaje sostiene la afirmación?, ¿se respetaron negaciones/condiciones?, ¿qué quedó persistido?

El PDF de la sección 10 recoge los resultados que existían al ejecutarla; si después ejecutas OCR,
vuelve a ejecutar solo la sección de PDF para incluirlo. No repetir la consulta Groq innecesariamente.
La entrega de clase 08 es incremental y no sustituye la evaluación completa de clase 10.

In [ ]:
flags={name:value for name,value in list(globals().items()) if name in (
    'ENABLE_EMBEDDINGS','CONNECT_SUPABASE','UPLOAD_EXISTING_INDEX','RUN_GROQ_QUERY',
    'RUN_CLASS08_CASES','RUN_REMOTE_REGRESSION','COMPARE_CHUNKING','RUN_OCR','RUN_API_CHECKS',
    'RUN_LOCAL_TESTS','GENERATE_PDF','RUN_LOCAL_MOCK')}
closure={
    'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'notebooks/clase_08_rag_supabase_groq.ipynb',
    'corpus_id':CORPUS_ID,'corpus_simulado':True,
    'prompts':{'classification':PROMPT_PATH.name,'documentary':RAGConfig().prompt.name},
    'flags':flags,
    'results':[{'case_id':r['case_id'],'source':r['source'],'state':r['state'],
                'citations_valid':r['rag']['citations_valid'],
                'support_review':r['rag']['semantic_support_review']} for _,r in observed_results],
    'persistence_checked_this_session':persistence_result is not None,
    'api_checked_this_session':api_result is not None,
    'manual_review':'PENDIENTE: anotar afirmaciones respaldadas, acierto y límite antes de entregar.',
    'pending':['Suite remota completa con configuración final y cuota disponible',
               'Aislamiento RLS entre dos usuarios Auth reales',
               'Justificación humana de fragmentación y revisión de respuestas'],
}
display(closure)
print('Bitácora guardada:',save_json('bitacora.json',closure))